In [1]:
import json
import shutil
from pathlib import Path

# ========== 配置路径 ==========
base_dir = Path("/home/shaolingxuan/project/WebThinker/outputs/gaia.qwen3.5-9b.webthinker")
source_json = base_dir / "GAIA_text.json"
all_jsonl = base_dir / "all.jsonl"
backup_jsonl = base_dir / "all.jsonl.bak"

# ========== 1. 读取 GAIA_text.json（完整数据） ==========
print(f"【读取 {source_json.name}】")
with open(source_json, "r", encoding="utf-8") as f:
    first_char = f.read(1)

if first_char == '[':
    with open(source_json, "r", encoding="utf-8") as f:
        source_records = json.load(f)
    print(f"  ✅ 成功读取 {len(source_records)} 条记录")
else:
    source_records = []
    with open(source_json, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                source_records.append(json.loads(line))
    print(f"  ✅ 成功读取 {len(source_records)} 条记录")

# 建立映射: 把所有形式的 ID 都作为 key 存下来
id_to_question = {}
for item in source_records:
    question = item.get("Question") or item.get("question") or item.get("query")
    if not question:
        continue
    
    # 无论是 id 还是 task_id，都存入字典，防止遗漏
    for k in ["id", "ID", "Id", "task_id"]:
        if k in item and item[k] is not None:
            id_to_question[str(item[k])] = question

print(f"  ✅ 建立 {len(id_to_question)} 条 id→Question 映射 (包含多重ID映射)")


# ========== 2. 读取 all.jsonl（需要补全的文件） ==========
print(f"\n【读取 {all_jsonl.name}】")
all_records = []
with open(all_jsonl, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            all_records.append(json.loads(line))

print(f"  ✅ 成功读取 {len(all_records)} 条记录")


# ========== 3. 补充 Question 字段 ==========
print("\n【补充 Question 字段】")
updated_count = 0
not_found_count = 0
already_has_count = 0

for i, item in enumerate(all_records):
    # 这里我们优先取 id，取不到再取 task_id
    qid = item.get("id") or item.get("ID") or item.get("Id") or item.get("task_id")
    qid_str = str(qid) if qid is not None else None

    # 如果已经有 Question 字段就跳过
    if "Question" in item or "question" in item:
        already_has_count += 1
        continue

    if qid_str is not None and qid_str in id_to_question:
        item["Question"] = id_to_question[qid_str]
        updated_count += 1
    else:
        not_found_count += 1
        print(f"  [!] 第 {i+1} 行 (id={qid}) 未找到匹配的 Question")


# ========== 4. 备份并写回 ==========
if not backup_jsonl.exists():
    shutil.copy2(all_jsonl, backup_jsonl)
    print(f"\n✅ 已备份原始文件到: {backup_jsonl}")
else:
    print(f"\n✅ 备份文件已存在: {backup_jsonl}（跳过备份）")

with open(all_jsonl, "w", encoding="utf-8") as f:
    for item in all_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n{'=' * 60}")
print(f"  处理完成!")
print(f"  总记录数:           {len(all_records)}")
print(f"  本次新增 Question:  {updated_count}")
print(f"  原本已有 Question:  {already_has_count}")
print(f"  未找到匹配 id:      {not_found_count}")
print(f"{'=' * 60}")

【读取 GAIA_text.json】
  ✅ 成功读取 103 条记录
  ✅ 建立 206 条 id→Question 映射 (包含多重ID映射)

【读取 all.jsonl】
  ✅ 成功读取 103 条记录

【补充 Question 字段】

✅ 备份文件已存在: /home/shaolingxuan/project/WebThinker/outputs/gaia.qwen3.5-9b.webthinker/all.jsonl.bak（跳过备份）

  处理完成!
  总记录数:           103
  本次新增 Question:  67
  原本已有 Question:  36
  未找到匹配 id:      0
